# Evaluating Multiple LM Outputs (External)

In [ ]:
# imports
import json
import pandas as pd
import importlib.util
import sys
from os.path import join
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    format_features, format_model_info
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion
from stat_genie.blade_pipeline.additions.eval.judge import run_judge_evaluation

In [30]:
# load files
output_dir = "/accounts/projects/binyu/hao_huang/stat-genie/examples/output"
analysis_subdir_path_1 = f"{output_dir}/soccer_default"
analysis_subdir_path_2 = f"{output_dir}/soccer_perturb_feature_names"

# CHANGE THIS PATH TO WHERE YOU WANT TO SAVE THE EVALUATION RESULTS
output_save_path = f"{output_dir}/external_eval_results.json"
multirun_filename_1 = "multirun_analyses.json"
multirun_filename_2 = "multirun_analyses.json"

# use both files to get analysis code paths
multirun_path_1 = join(analysis_subdir_path_1, multirun_filename_1)
multirun_path_2 = join(analysis_subdir_path_2, multirun_filename_2)

with open(multirun_path_1, "r") as file:
    multirun_analyses_1 = json.load(file)

with open(multirun_path_2, "r") as file:
    multirun_analyses_2 = json.load(file)

num_analyses_1 = multirun_analyses_1['n']
num_analyses_2 = multirun_analyses_2['n']

analysis_code_filenames_1 = [f"llm_analysis_{i}.py" for i in range(num_analyses_1)]
analysis_code_filenames_2 = [f"llm_analysis_{i}.py" for i in range(num_analyses_2)]

analysis_code_paths_1 = [join(analysis_subdir_path_1, filename)
                         for filename in analysis_code_filenames_1]

analysis_code_paths_2 = [join(analysis_subdir_path_2, filename)
                         for filename in analysis_code_filenames_2]

In [31]:
llm_provider = "openai"
llm_model = "gpt-5-mini"
llm_assistant = llm(provider=llm_provider, model=llm_model)

[2025-12-04 07:16:09.37][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/projects/binyu/hao_huang/stat-genie/config/llm_eval_config.yml'.


In [32]:
features_1 = format_features(multirun_analyses_1, num_analyses_1, llm_assistant)

[2025-12-04 07:16:09.96][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:16:15.73][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  5.77 seconds
[2025-12-04 07:16:15.77][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 07:16:15.81][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:16:19.31][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  3.50 seconds
[2025-12-04 07:16:19.31][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 07:16:19.36][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:16:28.10][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  8.73 

In [33]:
features_2 = format_features(multirun_analyses_2, num_analyses_2, llm_assistant)

In [34]:
model_info_1 = format_model_info(multirun_analyses_1, num_analyses_1, llm_assistant)

[2025-12-04 07:19:51.21][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:20:06.38][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  15.16 seconds
[2025-12-04 07:20:06.39][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 07:20:06.41][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:20:14.72][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  8.32 seconds
[2025-12-04 07:20:14.73][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 07:20:14.75][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:20:29.64][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  14.9

In [35]:
model_info_2 = format_model_info(multirun_analyses_2, num_analyses_2, llm_assistant)

In [36]:
# load dataset, need more user-friendly input method later
dataset_name = multirun_analyses_1['dataset_name']
dataset_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                    "datasets", dataset_name, "data.csv")

data = pd.read_csv(dataset_path)

In [37]:
transform_functions_1 = {}
transform_functions_2 = {}
model_functions_1 = {}
model_functions_2 = {}

# ----- Loop for first set -----
for i, analysis_code_path in enumerate(analysis_code_paths_1):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_{i}"] = module
    spec.loader.exec_module(module)
    
    transform_functions_1[i] = module.transform
    model_functions_1[i] = module.model

# ----- Loop for second set -----
for i, analysis_code_path in enumerate(analysis_code_paths_2):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_2_{i}"] = module
    spec.loader.exec_module(module)

    transform_functions_2[i] = module.transform
    model_functions_2[i] = module.model

In [38]:
transformed_datasets_1 = {}
for i, transform_func in transform_functions_1.items():
    try:
        transformed_datasets_1[i] = transform_func(data.copy())
        print(f"[Transform 1-{i}] ✅ Completed successfully.")
    except Exception:
        print(f"[Transform 1-{i}] ❌")
        transformed_datasets_1[i] = None

model_results_1 = {}
for i, model_func in model_functions_1.items():
    try:
        if transformed_datasets_1[i] is None:
            print(f"[Model 1-{i}] ⚠️ Skipping — transform failed.")
            continue

        model_results_1[i] = model_func(transformed_datasets_1[i].copy())
        print(f"[Model 1-{i}] ✅ Completed successfully.")
    except Exception as e:
        print(f"[Model 1-{i}] ❌ Failed with error: {e}")
        model_results_1[i] = None

[Transform 1-0] ✅ Completed successfully.
[Transform 1-1] ✅ Completed successfully.
[Transform 1-2] ✅ Completed successfully.
[Model 1-0] ✅ Completed successfully.


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 1-1] ✅ Completed successfully.


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 1-2] ✅ Completed successfully.


In [39]:
transformed_datasets_2 = {}
for i, transform_func in transform_functions_2.items():
    try:
        transformed_datasets_2[i] = transform_func(data.copy())
        print(f"[Transform 2-{i}] ✅ Completed successfully.")
    except Exception:
        print(f"[Transform 2-{i}] ❌")
        transformed_datasets_2[i] = None

model_results_2 = {}
for i, model_func in model_functions_2.items():
    try:
        if transformed_datasets_2[i] is None:
            print(f"[Model 2-{i}] ⚠️ Skipping — transform failed.")
            continue

        model_results_2[i] = model_func(transformed_datasets_2[i].copy())
        print(f"[Model 2-{i}] ✅ Completed successfully.")
    except Exception as e:
        print(f"[Model 2-{i}] ❌ Failed with error: {e}")
        model_results_2[i] = None

[Transform 2-0] ✅ Completed successfully.
[Transform 2-1] ✅ Completed successfully.
[Transform 2-2] ✅ Completed successfully.


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 2-0] ✅ Completed successfully.


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 2-1] ✅ Completed successfully.


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 2-2] ✅ Completed successfully.


In [40]:
final_answer_code_1 = {}
final_answer_code_2 = {}

info_json_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                      "datasets", dataset_name, "info.json")
with open(info_json_path, "r") as file:
    info_json = json.load(file)

task = info_json['research_questions']

for i in range(num_analyses_1):

    independent_variable = features_1[i]['independent_variables']
    dependent_variable = features_1[i]['response_variables']

    model_code = multirun_analyses_1['analyses'][str(i)]['m_code']
    model_output = model_results_1[i]

    final_answer_code_1[i] = write_final_answer_code(
        llm_assistant=llm_assistant,
        task=task,
        independent_variable=independent_variable,
        dependent_variable=dependent_variable,
        model_code=model_code,
        model_output=model_output,
    )

[2025-12-04 07:22:26.01][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:23:08.68][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  42.67 seconds
[2025-12-04 07:23:08.68][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 07:23:08.70][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:23:34.09][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  25.39 seconds
[2025-12-04 07:23:34.09][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 07:23:34.12][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:24:17.04][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  42.

In [41]:
for i in range(num_analyses_2):

    independent_variable = features_2[i]['independent_variables']
    dependent_variable = features_2[i]['response_variables']

    model_code = multirun_analyses_2['analyses'][str(i)]['m_code']
    model_output = model_results_2[i]

    final_answer_code_2[i] = write_final_answer_code(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        model_output,
    )

[2025-12-04 07:24:17.18][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:24:47.93][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  30.75 seconds
[2025-12-04 07:24:47.94][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 07:24:48.01][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:25:25.31][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  37.30 seconds
[2025-12-04 07:25:25.32][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 07:25:25.36][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:26:06.00][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  40.

In [42]:
print(final_answer_code_1)
print(final_answer_code_2)

{0: 'def extract_final_answer(model_output):\n    """\n    Extracts the effect of DarkSkin from a fitted statsmodels GLM/GLMResults-like object.\n    Returns a dict with keys:\n      - "object": a dict of extracted statistics (coef, se, pvalue, CI, IRR, IRR_CI, model info, boolean conclusion)\n      - "description": a short plain-language interpretation answering whether dark-skinned players\n                       are more likely to receive red cards.\n    """\n    import numpy as np\n\n    result = {}\n    try:\n        res = model_output  # expected statsmodels results wrapper\n\n        # Find the parameter name corresponding to DarkSkin (robust to small naming differences)\n        params_index = list(res.params.index) if hasattr(res, "params") else []\n        param_name = None\n        for name in params_index:\n            if name == "DarkSkin" or name.endswith(".DarkSkin") or "DarkSkin" in name:\n                param_name = name\n                break\n        if param_name i

In [43]:
final_answer_functions_1 = {}

for i in range(num_analyses_1):
    namespace = {}

    compiled_code = compile(
        final_answer_code_1[i],
        f"<final_answer_code_1_{i}>",
        "exec"
    )
    exec(compiled_code, namespace)

    final_answer_functions_1[i] = namespace['extract_final_answer']

final_answers_1 = [
    final_answer_functions_1[i](model_results_1[i])
    for i in range(num_analyses_1)
]

In [44]:
final_answer_functions_2 = {}

for i in range(num_analyses_2):
    namespace = {}

    compiled_code = compile(
        final_answer_code_2[i],
        f"<final_answer_code_2_{i}>",
        "exec"
    )
    exec(compiled_code, namespace)

    final_answer_functions_2[i] = namespace['extract_final_answer']

final_answers_2 = [
    final_answer_functions_2[i](model_results_2[i])
    for i in range(num_analyses_2)
]

In [45]:
conclusions_1 = {}

for i in range(num_analyses_1):
    independent_variable = features_1[i]['independent_variables']
    dependent_variable = features_1[i]['response_variables']

    model_code = multirun_analyses_1['analyses'][str(i)]['m_code']
    interpretation_code = final_answer_code_1.get(i, None)
    interpretation_output = final_answers_1[i] if i < len(final_answers_1) else None

    conclusions_1[i] = make_conclusion(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        interpretation_code,
        interpretation_output
    )



[2025-12-04 07:26:06.51][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:26:13.60][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.09 seconds
[2025-12-04 07:26:13.60][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 07:26:13.64][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:26:18.39][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  4.75 seconds
[2025-12-04 07:26:18.39][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 07:26:18.43][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:26:24.39][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  5.96 

In [46]:
conclusions_2 = {}

for i in range(num_analyses_2):
    independent_variable = features_2[i]['independent_variables']
    dependent_variable = features_2[i]['response_variables']

    model_code = multirun_analyses_2['analyses'][str(i)]['m_code']

    interpretation_code = final_answer_code_2.get(i, None)
    interpretation_output = final_answers_2[i] if i < len(final_answers_2) else None

    conclusions_2[i] = make_conclusion(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        interpretation_code,
        interpretation_output
    )

[2025-12-04 07:26:24.61][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:26:29.72][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  5.12 seconds
[2025-12-04 07:26:29.73][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 07:26:29.74][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:26:34.09][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  4.34 seconds
[2025-12-04 07:26:34.09][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 07:26:34.13][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:26:39.85][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  5.72 

In [47]:
data_head = data.head(10)

In [48]:
print()

In [76]:
import importlib
import stat_genie.blade_pipeline.additions.eval.judge as judge
import importlib
importlib.reload(judge)


<module 'stat_genie.blade_pipeline.additions.eval.judge' from '/accounts/projects/binyu/hao_huang/stat-genie/src/stat_genie/blade_pipeline/additions/eval/judge.py'>

In [77]:
results = judge.run_judge_evaluation_pairwise(
    task,
    data_head,
    features_1, features_2,
    model_info_1, model_info_2,
    conclusions_1, conclusions_2,
    llm_provider="openai",
    llm_model="gpt-5-mini",
    output_path="pairwise_results.json"
)


[2025-12-04 07:44:57.13][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/projects/binyu/hao_huang/stat-genie/config/llm_eval_config.yml'.
[2025-12-04 07:44:57.72][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:45:10.92][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  13.20 seconds
[2025-12-04 07:45:10.93][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 07:45:11.00][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 07:45:21.31][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  10.31 seconds
[2025-12-04 07:45:21.32][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 07:45:21.38][base.py:60 - stat_genie.blade_pip

In [ ]:
judge_results